In [2]:
## Imports

import sys, os
from pathlib import Path

parent_folder = str(Path.cwd().parents[0])
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

from sigpy import mri
import scipy
import pickle
from sklearn.decomposition import PCA
import seaborn as sns
import sigpy as sp
import cupy as cp
import numpy as np
from scipy.io import savemat
import twixtools
import matplotlib.pyplot as plt
import pickle_utils
import mapvbvd

In [3]:
data_file = '/data/lilianae/NaF_Patient3/twix/anon_meas_MID00283_FID65311_Tho_fl3d_star_vibe_991_nav_tj_2000sp_AllCoils_SOS.dat'
# import mapvbvd

# twixObj = mapvbvd.mapVBVD(data_file)
# twix_img = twixObj.image['']
# ncoil = 15
# ksp_resp = []
# for p in range(ncoil):
#     ksp_tmp=twix_img[:,p,:,:,:,:,:,0]
#     ksp_resp.append(ksp_tmp[247:267,np.newaxis,:,19:26])

multi_twix = twixtools.read_twix(str(data_file))

Software version: VD/VE (!?)

Scan  0


100%|██████████| 15.9G/15.9G [00:15<00:00, 1.12GB/s]


In [4]:
from twixtools.recon_helpers import remove_oversampling
# chronological_data = []
mdb_list_for_dc = []

for i, mdb in enumerate(multi_twix[-1]['mdb']):
## Use same logic as twix_category['image'] to get mdh values for k-space
    if (not mdb.is_flag_set('SYNCDATA') and
        not mdb.is_flag_set('ACQEND') and
        not mdb.is_flag_set('RTFEEDBACK') and
        not mdb.is_flag_set('HPFEEDBACK') and
        not mdb.is_flag_set('REFPHASESTABSCAN') and
        not mdb.is_flag_set('PHASESTABSCAN') and
        not mdb.is_flag_set('PHASCOR') and
        not mdb.is_flag_set('NOISEADJSCAN') and
        not mdb.is_flag_set('noname60') and
        (not mdb.is_flag_set('PATREFSCAN') or mdb.is_flag_set('PATREFANDIMASCAN'))):

        if not np.isnan(mdb.mdh.TimeStamp):
            ## Extract k-space data for this readout
            mdb_data = mdb.data  # Shape: (channels, samples)

            ## Apply oversampling removal to ensure consistent array sizes
            if mdb_data.shape[-1] == 512:  ## if we have 512 points, this is an image line. If there are 704 points, it is a noise scan. Discovered from manual inspection

                # mdb_data, _ = remove_oversampling(mdb_data, x_was_in_timedomain=True)
                mdb_data = mdb_data    ## Only take first 256 samples, same as logic Michael used in former data processing code
                mdb_list_for_dc.append(mdb)
                
                
# Filter to first echo only, matching MATLAB's echo=1 selection
mdb_list_for_dc_echo0 = [mdb for mdb in mdb_list_for_dc if mdb.mdh.Counter.Eco == 0]
# mdb_list_for_dc_echo0 = [mdb for mdb in mdb_list_for_dc]

In [10]:
# print(mdb_list_for_dc_echo0)
print(mdb_list_for_dc_echo0[0].data.shape)

(15, 512)


In [7]:
384/192

2.0

In [6]:
# 1. Get the target value from MATLAB (Coil 2 in MATLAB is index 1)
# You established py_data[0,0] matches mat_raw(1,1,1,1,1)
# Now we want to find where mat_raw(1,2,1,1,1) exists in Python
target_val = 8.6822547e-07 + 9.7136945e-07j

# 2. Search through MDBs to find the offset
for i, mdb in enumerate(mdb_list_for_dc_echo0):
    # mdb.data shape is (15, 512)
    # Let's check the first 704 potential spots in the raw buffer memory
    # Since Coil 1 is offset in the file, it might actually be sitting 
    # inside the 'Coil 0' array of your Python MDB if the length is 704
    # print(mdb)
    for coil_idx in range(mdb.data.shape[0]):
        for sample_idx in range(mdb.data.shape[1]):
            val = mdb.data[coil_idx, sample_idx]
            
            if np.allclose(val, target_val):
                print(f"MATCH FOUND!")
                print(f"MDB Index: {i}")
                print(f"Python Coil Index: {coil_idx}")
                print(f"Python Sample Index: {sample_idx}")


MATCH FOUND!
MDB Index: 0
Python Coil Index: 2
Python Sample Index: 384


KeyboardInterrupt: 

In [ ]:
# def import_kspace(image_mdbs):
#     n_line = 1 + max([mdb.cLin for mdb in image_mdbs])
#     n_part = 1 + max([mdb.cPar for mdb in image_mdbs])
#     n_channel, n_column = image_mdbs[0].data.shape

#     out = np.zeros([n_part, n_line, n_channel, n_column], dtype=np.complex64)
#     counts = np.zeros([n_part, n_line]) 

#     for mdb in image_mdbs:
#         out[mdb.cPar, mdb.cLin] += mdb.data # Accumulate
#         counts[mdb.cPar, mdb.cLin] += 1

#     # Divide by counts to get the mean (averaging)
#     # This prevents the "last MDB wins" overwriting problem
#     out /= np.maximum(counts[:, :, np.newaxis, np.newaxis], 1)
#     return out

##read image data from list of mdbs and sort into 3d k-space (+ coil dim.)
# def import_kspace(image_mdbs):

#     n_line = 1 + max([mdb.cLin for mdb in image_mdbs])
#     n_part = 1 + max([mdb.cPar for mdb in image_mdbs])
#     n_channel, n_column = image_mdbs[0].data.shape

#     out = np.zeros([n_part, n_line, n_channel, n_column], dtype=np.complex64)
#     for mdb in image_mdbs:
#         # '+=' takes care of averaging, but careful in case of other counters (e.g. echoes)
#         out[mdb.cPar, mdb.cLin] = mdb.data

#     return out  # 4D numpy array [n_part, n_line, n_channel, n_column]

def import_kspace(image_mdbs):
    # 1. Determine dimensions from the loop-level attributes
    n_line = 1 + max([mdb.cLin for mdb in image_mdbs])
    n_part = 1 + max([mdb.cPar for mdb in image_mdbs])
    
    # 2. Get channel count and columns directly from the data shape (15, 512)
    n_total_channels, n_column = image_mdbs[0].data.shape
    
    # 3. Initialize the 4D k-space array
    # Shape: [Partitions, Lines, Channels, Columns]
    out = np.zeros([n_part, n_line, n_total_channels, n_column], dtype=np.complex64)
    
    for mdb in image_mdbs:
        # Since each MDB contains all 15 channels, we can insert the 
        # entire (15, 512) array into the correct Partition and Line slot.
        # We transpose or slice carefully to match (Channels, Columns)
        out[mdb.cPar, mdb.cLin, :, :] = mdb.data

    return out

def import_kspace_aligned(image_mdbs):
    n_line = 1 + max([mdb.cLin for mdb in image_mdbs])
    n_part = 1 + max([mdb.cPar for mdb in image_mdbs])
    
    # We initialize with 704 columns to match MATLAB's stride
    out = np.zeros([n_part, n_line, 15, 704], dtype=np.complex64)
    
    for mdb in image_mdbs:
        # 1. Flatten the (15, 512) data into one long 1D array (7,680 samples)
        # This is the "ribbon" of data as Python sees it.
        flat_ribbon = mdb.data.flatten()
        
        # 2. Re-cut the ribbon every 704 samples
        # We can only do this for as many coils as fit in the 7,680 samples.
        # 7680 / 704 = 10.9 coils. (This means coils 11-14 are LOST in this MDB)
        for c in range(10): 
            start = c * 704
            end = start + 704
            
            # Extract the TRUE 704 samples for this specific coil
            true_coil_data = flat_ribbon[start:end]
            
            # 3. Place it in the out array
            out[mdb.cPar, mdb.cLin, c, :] = true_coil_data

    return out

out = import_kspace_aligned(mdb_list_for_dc)
print(f'out.shape = {out.shape}')

In [ ]:
# MDB_No = 10
# image_mdbs = mdb_list_for_dc_echo0
# channels_ID = [ch_hdr.ChannelId for ch_hdr in image_mdbs[MDB_No].channel_hdr]
# print(channels_ID)

In [ ]:
# pickle_utils.write_pickle(out, 'TEST_kspace_for_dc_patient3_mid0283.pkl')

In [ ]:
ksp_tmp_py= out
# ksp_tmp_py = pickle_utils.read_pickle('/home/lilianae/projects/naf_clean/load_data_pipeline/TEST_kspace_for_dc_patient3_mid0283.pkl')
print(ksp_tmp_py.shape)
n_slices, n_spokes, n_coils, n_ro = ksp_tmp_py.shape
ksp_resp_py_list = []
for coil in range(n_coils):
    ksp_resp_coil = ksp_tmp_py[19:26, :, coil, 247:267]
    ksp_resp_py_list.append(ksp_resp_coil)

ksp_resp_py = np.stack(ksp_resp_py_list, axis=0)
ksp_resp_py = np.transpose(ksp_resp_py, (3, 0, 2, 1))
print(f'ksp_resp_py.shape = {ksp_resp_py.shape}')

In [ ]:
from scipy.io import loadmat

data = loadmat('comparison_subject3/resp_signal_all_vars_subject3_mid0283.mat')

ksp_tmp_mat = data['ksp_tmp'].squeeze()
ksp_resp_mat = data['ksp_resp']

In [ ]:
# print(twix_image.image)
print(f'PYTHON \n')
print(f'ksp_resp_py.shape = {ksp_resp_py.shape}')

print(f'MATLAB \n')
print(f'ksp_tmp_mat.shape = {ksp_tmp_mat.shape}')
print(f'ksp_resp_mat.shape = {ksp_resp_mat.shape}')

In [ ]:
# Check every possible coil mapping
for py_coil in range(15):
    for mat_coil in range(15):
        if np.allclose(ksp_resp_py[:, py_coil, :, :], ksp_resp_mat[:, mat_coil, :, :]):
            print(f"Python coil {py_coil} matches MATLAB coil {mat_coil}")

In [ ]:
# Check if MATLAB and Python coil 1 match Python coil 0 data
print(np.allclose(ksp_resp_py[:, 0, :, :], ksp_resp_mat[:, 0, :, :]))  # True
print(np.allclose(ksp_resp_py[:, 1, :, :], ksp_resp_mat[:, 1, :, :]))  # likely False

# Check if coils are simply shifted by 1
print(np.allclose(ksp_resp_py[:, 0, :, :], ksp_resp_mat[:, 1, :, :]))  # test shift

In [ ]:
# import scipy.io

# def load_mat_variables(filepath: str) -> dict:
#     """
#     Load all user-defined variables from a .mat file.
    
#     Filters out scipy.io metadata keys (those starting/ending with '__').
    
#     Args:
#         filepath: Path to the .mat file.
    
#     Returns:
#         A dictionary of variable names to their values.
#     """
#     mat_data = scipy.io.loadmat(filepath)
#     return {k: v for k, v in mat_data.items() if not k.startswith("__") and not k.endswith("__")}

# variables = load_mat_variables('comparison_subject3/resp_signal_all_vars_subject3_mid0283.mat')

# for name, value in variables.items():
#     print(f"{name}: {value}")


In [ ]:
from scipy.io import loadmat

data = loadmat('comparison_subject3/resp_signal_all_vars_subject3_mid0283.mat')

twix_image= data['twix_obj']
ksp_tmp_mat = data['ksp_tmp'].squeeze()
ksp_resp_mat = data['ksp_resp']
resp_tmp_mat = data['resp_tmp'].squeeze()
filt_mat = data['filt'].squeeze()
resp_ft_filt_mat = data['resp_ft_filt'].squeeze()
# resp_signal_sort_mat = data['resp_signal_sort'].squeeze()
# resp_signal_mat = data['resp_signal'].squeeze()

# print(f'MATLAB:')
# print(f'ksp_tksp_tmp_mat.shape)
# print(resp_signal_mat.shape)

In [ ]:
# print(twix_image.image)
print(f'PYTHON \n')
print(f'ksp_resp_py.shape = {ksp_resp_py.shape}')

print(f'MATLAB \n')
print(f'ksp_tmp_mat.shape = {ksp_tmp_mat.shape}')
print(f'ksp_resp_mat.shape = {ksp_resp_mat.shape}')

In [ ]:
# Check if MATLAB and Python coil 1 match Python coil 0 data
print(np.allclose(ksp_resp_py[:, 0, :, :], ksp_resp_mat[:, 0, :, :]))  # True
print(np.allclose(ksp_resp_py[:, 1, :, :], ksp_resp_mat[:, 1, :, :]))  # likely False

# Check if coils are simply shifted by 1
print(np.allclose(ksp_resp_py[:, 0, :, :], ksp_resp_mat[:, 1, :, :]))  # test shift

In [ ]:
# Check every possible coil mapping
for py_coil in range(15):
    for mat_coil in range(15):
        if np.allclose(ksp_resp_py[:, py_coil, :, :], ksp_resp_mat[:, mat_coil, :, :]):
            print(f"Python coil {py_coil} matches MATLAB coil {mat_coil}")

In [ ]:
ksp_diff = ksp_resp_py[:, 1, :, :] - ksp_resp_mat[:, 1, :, :]
print(ksp_diff)

Compare center_of_kspace and ksp_resp

In [ ]:
import numpy as np

def compare_signal_arrays(a: np.ndarray, b: np.ndarray, rtol: float = 1e-5, atol: float = 1e-8) -> None:
    """
    Compare two arrays of physical signal data expected to be identical.

    Args:
        a, b:  Arrays to compare, expected shape (20, 15, 2002, 7).
        rtol:  Relative tolerance for np.allclose.
        atol:  Absolute tolerance for np.allclose.
    """
    print("=" * 50)

    # --- Shape ---
    print(f"Shape A: {a.shape} | Shape B: {b.shape}")
    if a.shape != b.shape:
        print("FAIL: shapes differ — aborting further checks.")
        return

    # --- Dtype ---
    print(f"Dtype A: {a.dtype} | Dtype B: {b.dtype}")

    # --- NaN / Inf ---
    nan_a, nan_b = np.isnan(a).sum(), np.isnan(b).sum()
    inf_a, inf_b = np.isinf(a).sum(), np.isinf(b).sum()
    print(f"NaNs  — A: {nan_a} | B: {nan_b}")
    print(f"Infs  — A: {inf_a} | B: {inf_b}")

    # --- Exact equality ---
    exact_match = np.array_equal(a, b)
    print(f"Exact match (array_equal): {exact_match}")

    # --- Numeric closeness ---
    close_match = np.allclose(a, b, rtol=rtol, atol=atol, equal_nan=True)
    print(f"Close match (allclose, rtol={rtol}, atol={atol}): {close_match}")

    # --- Difference statistics ---
    diff = np.abs(a.astype(np.float64) - b.astype(np.float64))
    print(f"Max absolute diff  : {diff.max():.6e}")
    print(f"Mean absolute diff : {diff.mean():.6e}")
    print(f"Std of diff        : {diff.std():.6e}")

    # --- Locate worst mismatches ---
    n_worst = 5
    flat_idx = np.argpartition(diff.ravel(), -n_worst)[-n_worst:]
    worst_idx = np.unravel_index(flat_idx, diff.shape)
    print(f"\nTop {n_worst} worst mismatch locations (sample, channel, timestep, feature):")
    for i in zip(*worst_idx):
        print(f"  index {i} → A={a[i]:.6e}, B={b[i]:.6e}, diff={diff[i]:.6e}")

    # --- Per-dimension breakdown ---
    print(f"\nMax diff per axis-0 slice (20 samples) : {diff.max(axis=(1,2,3))}")
    print(f"Max diff per axis-1 slice (15 channels): {diff.max(axis=(0,2,3))}")
    print(f"Max diff per axis-3 slice (7 features) : {diff.max(axis=(0,1,2))}")

    print("=" * 50)

In [ ]:
compare_signal_arrays(ksp_resp_py, ksp_resp_mat)

In [ ]:
# print(twix_image.image)
print(f'PYTHON \n')
print(f'ksp_resp_py.shape = {ksp_resp_py.shape}')

print(f'MATLAB \n')
print(f'ksp_tmp_mat.shape = {ksp_tmp_mat.shape}')
print(f'ksp_resp_mat.shape = {ksp_resp_mat.shape}')

In [ ]:
resp_tmp = np.squeeze(np.sum(np.sum(np.abs(ksp_resp_py), axis=4), axis=1))
print(resp_tmp.shape)

Compare resp_tmp

In [ ]:
import numpy.fft as fft

n_coils = resp_tmp.shape[0]
n_slices_acq = 58
TR = 6
n_spokes = 2002

resp_ft = np.zeros_like(resp_tmp, dtype=np.complex128)

for coil in range(n_coils):
    resp_ft[coil,:] = fft.fftshift(fft.fft(resp_tmp[coil,:]) )

## Create frequency axis
f_sampling = 1 /((TR*1e-3)*n_slices_acq)
f_max = f_sampling/2
f_step = f_max / 1000

f_axis = np.linspace(-f_max, f_max + f_step, resp_ft.shape[1])

f_ga = (111.246 / 360) / (TR*1e-3 * n_slices_acq)
fw = 50 # Filter linewidth

## Generate Gaussian band-stop filter
filt = np.exp(-(fw * (f_axis - f_ga))**2)
filt2 = filt + np.flip(filt)
resp_ft_filt = resp_ft * (1-filt2)

In [ ]:
n_spokes = 2002
apod_edge = 0.9
fdw = 0.05

f_half =  f_axis[len(f_axis)//2:]
filt_FD_half = 1/(1 + np.exp((f_half - apod_edge * f_max)/fdw))
filt_FD = np.concatenate([np.flip(filt_FD_half), filt_FD_half])


## Adjust length if number of spokes is odd
if len(filt_FD) > n_spokes:
    filt_FD = filt_FD[:n_spokes]

resp_ft_filt_apodized = resp_ft_filt * filt_FD
resp_tmp_filt = fft.ifft(fft.fftshift(resp_ft_filt_apodized, axes=1), axis=1) 

# 2. Convert to real magnitude and scale correctly
# Using np.abs() ensures it stays positive like the MATLAB trace
resp_magnitude_data = np.abs(resp_tmp_filt)

# 3. PCA on the scaled magnitude
pca = PCA(n_components=1)
resp_pca = pca.fit_transform(resp_magnitude_data.T) 
resp_signal_python = resp_pca[:, 0]

# 4. Re-insert the baseline
# resp_signal_python += np.mean(resp_magnitude_data)
# pca = PCA(n_components=1)
# resp_pca = pca.fit_transform(np.real(resp_tmp_filt.T))
# resp_signal_python = resp_pca[:, 0]
# resp_signal_python += np.mean(np.real(resp_tmp_filt))

In [ ]:
resp_signal_diff = resp_signal_python - resp_signal_mat
print(resp_signal_diff.max())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

# 1. Load MATLAB data (as provided in your snippet)
# data = loadmat('comparison_subject3/resp_signal_subject3_mid0283.mat')
# resp_signal_mat = data['resp_signal'].squeeze()

# 2. Prepare the signals
# Assuming 'resp_signal' is your Python-generated variable
resp_signal_py = resp_signal_python.squeeze() 

# 4. Plotting
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Subplot 1: Full Signals (showing why we need to zoom)
ax.plot(resp_signal_mat, label='MATLAB (Full)', alpha=0.5)
ax.plot(resp_signal_py, label='Python (Full)', alpha=0.5)
ax.set_title("Full Comparison (Notice Edge Spikes)")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# # Create a figure with 3 subplots
# fig, ax = plt.subplots(3, 1, figsize=(10, 12))

# # Plot 1: The Filter Shapes
# ax[0].plot(f_axis, 1-filt2, label='GA Notch Filter (Band-stop)', color='orange')
# ax[1].plot(f_axis, filt_FD, label='Fermi-Dirac (Low-pass)', color='green')
# ax[0].set_title("Frequency Filter Masks")
# ax[0].legend()

# # Plot 2: Before vs After Spectrum (Magnitude)
# # Plotting for the first coil (index 0)
# ax[1].plot(f_axis, np.abs(resp_ft[0, :]), label='Raw Spectrum', alpha=0.5)
# ax[1].plot(f_axis, np.abs(resp_ft_filt[0, :]), label='Filtered Spectrum', color='red')
# ax[1].set_title("Frequency Spectrum (Coil 0)")
# ax[1].set_yscale('log') # Log scale helps see the noise floor
# ax[1].legend()

# # Plot 3: The Filtered Trace in Time Domain
# ax[2].plot(resp_tmp_filt[0, :], label='Filtered Trace (Coil 0)')
# ax[2].set_title("Filtered Time-Domain Respiratory Trace")
# ax[2].set_xlabel("Spoke Index")
# ax[2].legend()

# plt.tight_layout()
# plt.show()

In [ ]:

# # 1. Clipping Buffer (Handle Gibbs Ringing from filtering)
# gibbs_factor = 0.08
# n_total = len(resp_signal_python)
# buffer = int(np.round(n_total * gibbs_factor / 2))

# # Extract the valid central portion of the signal
# resp_signal_clipped = resp_signal_python[buffer : n_total - buffer]

# # 2. Sort the signal and keep track of original indices
# # resp_ind are the indices relative to the clipped signal
# resp_ind = np.argsort(resp_signal_clipped)
# resp_signal_sort = resp_signal_clipped[resp_ind]

### Compare MATLAB and Python results

In [ ]:


# # 3. Identify Outliers (Deep Breaths/Coughs)
# signal_deep_breath = 0.021  # Threshold for rejection
# ind_deep_breath = np.where(resp_signal_sort < signal_deep_breath)[0]

# # 4. Define Gates
# n_gates = 5
# n_valid_samples = len(resp_signal_sort) - len(ind_deep_breath)
# samples_per_gate = n_valid_samples // n_gates

# # Create an array defining how many samples go into each gate
# n_samples_per_gate = np.full(n_gates, samples_per_gate)
# # Add deep breath samples to "Gate 1" (they will be moved to Gate 0 later)
# n_samples_per_gate[0] += len(ind_deep_breath)
# # Correct for rounding errors in the last gate
# n_samples_per_gate[-1] += (n_valid_samples % n_gates)

# # 5. Assign Gate Numbers
# # We use 0 for rejected data, 1 to n_gates for valid data
# resp_gates = np.zeros(n_total, dtype=int)
# current_pos = 0

# for n in range(1, n_gates + 1):
#     num_in_this_gate = n_samples_per_gate[n-1]
#     # Get the indices from the sorted list
#     indices_in_sort = resp_ind[current_pos : current_pos + num_in_this_gate]
#     # Map back to original acquisition index (adding buffer back)
#     resp_gates[indices_in_sort + buffer] = n
#     current_pos += num_in_this_gate

# # Re-assign the deep breath outliers specifically to Gate 0
# outlier_indices = resp_ind[ind_deep_breath]
# resp_gates[outlier_indices + buffer] = 0

# # 6. Visualization: Respiratory Trace with Gate Assignment (ZOOMED)
# plt.figure(figsize=(12, 5))
# acq_order = np.arange(n_total)

# # Plot the full line in the background
# plt.plot(acq_order, resp_signal, color='blue', alpha=0.3, label='Full Trace')

# colors = ['black', 'cyan', 'magenta', 'red', 'green', 'orange']
# labels = ['Rejected'] + [f'Gate {i+1}' for i in range(n_gates)]

# for g in range(n_gates + 1):
#     mask = (resp_gates == g)
#     plt.scatter(acq_order[mask], resp_signal[mask], color=colors[g], s=10, label=labels[g])

# # --- ZOOM LOGIC ---
# # 1. Zoom X-axis: Ignore the Gibbs buffer at start and end
# plt.xlim(buffer, n_total - buffer)

# # 2. Zoom Y-axis: Focus on the actual signal range in the middle
# valid_data = resp_signal[buffer : n_total - buffer]
# y_min, y_max = np.min(valid_data), np.max(valid_data)
# # Add a 10% margin so points aren't touching the edge of the box
# margin = (y_max - y_min) * 0.1
# plt.ylim(y_min - margin, y_max + margin)

# plt.xlabel('Acquisition Order (Spokes)')
# plt.ylabel('Respiratory Amplitude')
# plt.title('Respiratory Gating Results (Zoomed to Valid Signal)')
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.grid(True, alpha=0.2)
# plt.tight_layout()
# plt.show()